<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_00_environment_check.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · Notebook 00 — Environment Check

**Paired with L7.2 · Fundamental PDEs**

Time enters. Everything in Ex_07.1 was steady: one field, two coordinates, no
history. From here the network takes $(x, y, t)$ and the questions change —
how many conditions does a problem need, and what happens if you supply the
wrong number of them.

Run this first, top to bottom. Nothing to write.

## The two problems

| | | |
|---|---|---|
| **the die** | parabolic | first order in time — **one** initial condition |
| **the panel** | hyperbolic | second order in time — **two** initial conditions |

That difference is the subject of the whole exercise set, and notebook 04 makes
it cost something.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The two problems, in numbers

In [ ]:
pb.describe_problem()

**What you should see.**

```
  THE DIE  (parabolic, one initial condition)
    slow mode (1,1) : rate   17.37 1/s   tau   57.6 ms
    fast mode (2,1) : rate   43.43 1/s   tau   23.0 ms   (2.5x faster)

  THE PANEL  (hyperbolic, TWO initial conditions)
    fundamental     : 141.4 Hz   period 7.07 ms
    peak deflection : 0.563 mm
```

Two details worth pausing on.

**The die carries two modes, not one.** Mode $(m,n)$ on a square decays at
$(m^2+n^2)\pi^2\alpha/L^2$, so $(1,1)$ and $(2,1)$ give 2 and 5 — a ratio of
2.5. The field therefore does not simply shrink; it **changes shape** as the
sharp feature dies first. A model that matches the late field can still be
badly wrong early, which is a failure a single-mode problem cannot show you.

**The panel starts flat.** `u(x, y, 0) = 0` everywhere, and the energy is all
in the velocity. Hold that thought until notebook 04.

---

## 2 · Time is the last column

A convention, used everywhere in this course: space first, time last. So for a
point sampled as $(x, y, t)$, the time derivative is column 2.

In [ ]:
# u = sin(2x) * exp(-3t)   ->   u_t = -3u,  u_xx = -4u
pts = to_tensor(np.random.default_rng(0).uniform(0, 1, (200, 3)), requires_grad=True)
u = torch.sin(2 * pts[:, 0:1]) * torch.exp(-3 * pts[:, 2:3])

u_t = grad(u, pts)[:, 2:3]
u_xx = d2(u, pts, 0)

check("u_t  = -3u", to_numpy(u_t), -3.0 * to_numpy(u), tol=1e-9)
check("u_xx = -4u", to_numpy(u_xx), -4.0 * to_numpy(u), tol=1e-9)
print("\ngrad over three inputs:", tuple(grad(u, pts).shape), " -- column 2 is d/dt")

**What you should see.** Two `PASS` lines and `(200, 3)`.

A first-order-in-time residual needs `grad(...)[:, 2:3]`; a second-order one
needs `d2(u, xyt, 2)`. The die uses the first, the panel the second. Getting
the column wrong is a silent error — the code runs and solves a different
equation — so it is worth checking once, here, rather than debugging it later.

---

## 3 · Sampling a space–time slab

Three sets of points, three different jobs.

In [ ]:
xyt_f = spacetime_points(3000, pb.HEAT_DOMAIN, (0.0, pb.HEAT_T_END), seed=1)
xyt_0 = initial_points(400, pb.HEAT_DOMAIN, t0=0.0, seed=1)
xyt_b = boundary_points_in_time(25, 20, pb.HEAT_DOMAIN, (0.0, pb.HEAT_T_END), seed=1)

print("interior (PDE)      ", xyt_f.shape)
print("initial slice (IC)  ", xyt_0.shape, "   all t = 0:",
      bool((xyt_0[:, 2] == 0).all()))
print("boundary in time    ", xyt_b.shape, "   distinct instants:",
      len(np.unique(xyt_b[:, 2])))

fig = plt.figure(figsize=(7.6, 5.2))
ax = fig.add_subplot(projection="3d")
ax.scatter(xyt_f[::4, 0]*1e3, xyt_f[::4, 1]*1e3, xyt_f[::4, 2]*1e3,
           s=2, alpha=0.35, color="#1f77b4", label="interior")
ax.scatter(xyt_0[::2, 0]*1e3, xyt_0[::2, 1]*1e3, xyt_0[::2, 2]*1e3,
           s=6, color="#0f9d58", label="t = 0")
ax.scatter(xyt_b[::6, 0]*1e3, xyt_b[::6, 1]*1e3, xyt_b[::6, 2]*1e3,
           s=4, alpha=0.6, color="#d94f2b", label="edges")
ax.set_xlabel("x [mm]"); ax.set_ylabel("y [mm]"); ax.set_zlabel("t [ms]")
ax.set_title("The space–time slab, and what is sampled where")
ax.legend(fontsize=8, loc="upper left")
plt.show()

**What you should see.** A box of blue points, a green sheet at its base, and
orange points on its four vertical faces.

Note the shape of the problem. The initial condition lives on a **slice** —
one instant, the whole die — while the boundary condition lives on a **tube**,
four edges extruded through time. In Ex_07.1 both were curves; here they are
different kinds of object, and the number of points each needs is not the same.

---

## 4 · The exact solutions, checked

In [ ]:
X, Y, pts_np = grid_points(121, 121, pb.HEAT_DOMAIN)
x, y = pts_np[:, 0], pts_np[:, 1]

h, dt = 2e-5, 1e-6
for t in (0.0, 0.02, 0.10):
    u_t = (pb.heat_exact(x, y, t + dt) - pb.heat_exact(x, y, t - dt)) / (2 * dt)
    lap = ((pb.heat_exact(x + h, y, t) - 2*pb.heat_exact(x, y, t)
            + pb.heat_exact(x - h, y, t)) / h**2
           + (pb.heat_exact(x, y + h, t) - 2*pb.heat_exact(x, y, t)
              + pb.heat_exact(x, y - h, t)) / h**2)
    r = np.abs(u_t - pb.ALPHA * lap).max()
    print(f"  die   t = {t*1e3:5.1f} ms   max |u_t - a lap u| = {r:.2e}"
          f"   relative {r/np.abs(u_t).max():.1e}")

print()
Xp, Yp, pp = grid_points(121, 121, pb.WAVE_DOMAIN)
xp, yp = pp[:, 0], pp[:, 1]
hp, dtp = 1e-3, 1e-7
for t in (0.002, 0.010):
    u_tt = (pb.wave_exact(xp, yp, t + dtp) - 2*pb.wave_exact(xp, yp, t)
            + pb.wave_exact(xp, yp, t - dtp)) / dtp**2
    lap = ((pb.wave_exact(xp + hp, yp, t) - 2*pb.wave_exact(xp, yp, t)
            + pb.wave_exact(xp - hp, yp, t)) / hp**2
           + (pb.wave_exact(xp, yp + hp, t) - 2*pb.wave_exact(xp, yp, t)
              + pb.wave_exact(xp, yp - hp, t)) / hp**2)
    r = np.abs(u_tt - pb.C_WAVE**2 * lap).max()
    print(f"  panel t = {t*1e3:5.1f} ms   max |u_tt - c^2 lap u| = {r:.2e}"
          f"   relative {r/np.abs(u_tt).max():.1e}")

print()
print(f"  panel u(x,y,0)   max |.| = {np.abs(pb.wave_exact(xp, yp, 0.0)).max():.2e} m"
      "      <- flat")
print(f"  panel u_t(x,y,0) max |.| = {np.abs(pb.wave_velocity(xp, yp, 0.0)).max():.4f} m/s"
      "  <- moving")

**What you should see.** Relative residuals around 1e-6 — finite-difference
error, not a flaw in the solutions — and the two panel initial conditions:
**zero displacement, half a metre per second of velocity.**

---

## 5 · What the die actually does

In [ ]:
times = np.array([0.0, 0.01, 0.03, 0.08, 0.20])
fig, axes = plt.subplots(1, len(times), figsize=(16.0, 3.4))
for ax, t in zip(axes, times):
    pb.plot_slice(pb.heat_exact(x, y, t), pb.HEAT_DOMAIN, ax=ax,
                  title=f"t = {t*1e3:.0f} ms", label="θ [K]", scale=1e3)
plt.tight_layout(); plt.show()

k_slow, k_fast = pb.heat_rates()
tt = np.linspace(0, pb.HEAT_T_END, 300)
pb.plot_time_history(
    tt,
    {"broad mode (1,1)": pb.A_SLOW * np.exp(-k_slow * tt),
     "sharp mode (2,1)": pb.A_FAST * np.exp(-k_fast * tt),
     "ratio, sharp/broad": 20 * (pb.A_FAST/pb.A_SLOW) * np.exp(-(k_fast-k_slow) * tt)},
    title="Two timescales — the field changes shape as it cools",
    ylabel="amplitude [K]   (ratio ×20)")
plt.show()

**What you should see.** A two-lobed field at t = 0 that becomes single-lobed
and symmetric by 80 ms, and a ratio curve falling steadily toward zero.

That is the shape change. At t = 0 the sharp mode carries a third of the
amplitude; by 100 ms it carries almost none. **The die is not just getting
cooler, it is getting simpler**, and a model fitted mostly to late times will
have seen very little of the interesting part.

---

## 6 · Ready

Next: **notebook 01**, where the initial condition is a penalty in the loss and
the die is solved for the first time.